# 🌍 AI Travel Planner Agent with LangGraph

## Building an Intelligent Travel Planning System

Welcome to this comprehensive tutorial where we'll build an AI-powered travel planner using LangGraph! This agent will help users plan complete trips by coordinating multiple services and making intelligent decisions.

## 🎯 What We'll Build

Our travel planner will:
- ✈️ **Search for flights** and find the best options
- 🏨 **Find accommodations** based on preferences and budget
- 🌤️ **Check weather conditions** for the destination
- 🎭 **Discover local attractions** and activities
- 📅 **Create detailed itineraries** day by day
- 💰 **Manage budgets** and provide cost breakdowns
- 🔄 **Handle complex workflows** with conditional logic

## 🏗️ Architecture Overview

Our agent will use multiple specialized nodes:
1. **Input Processing** - Parse and validate user requests
2. **Flight Search** - Find and compare flight options
3. **Hotel Search** - Locate suitable accommodations
4. **Weather Check** - Get destination weather information
5. **Attractions Finder** - Discover local points of interest
6. **Itinerary Builder** - Create detailed daily plans
7. **Budget Calculator** - Provide cost analysis
8. **Plan Presenter** - Format and present the final plan

Let's start building! 🚀


In [1]:
from typing import Dict, List, Any, Optional, Annotated, TypedDict
from datetime import datetime, timedelta
import json
import random
from dataclasses import dataclass, asdict

In [2]:
from langgraph.graph import StateGraph, START, END


In [3]:
@dataclass
class TravelRequest:
    """User's travel request details"""
    origin: str
    destination: str
    departure_date: str  # YYYY-MM-DD
    return_date: Optional[str] = None  # YYYY-MM-DD, optional for one-way
    budget: Optional[float] = None  # User's budget for the trip
    preferences: Optional[List[str]] = None  # e.g., ["non-stop", "morning flight"]
    trip_type: Optional[str] = None  # "business, adventure, family, romantic, cultural, etc."

@dataclass
class FlightOption:
    """Details of a flight option"""
    airline: str
    departure_time: str
    arrival_time: str
    duration: str
    price: float
    stops: int

@dataclass
class HotelOption:
    """Hotel search Result"""
    name: str
    rating: str
    price_per_night: float
    amenities: List[str]
    location: str
    distance_from_centre: float


@dataclass
class WeatherInfo:
    """Weather information"""
    temperature_range: str
    conditions: str
    precipitation_chance: int
    recommendations: List[str]

@dataclass
class Attraction:
    """Tourist attraction information"""
    name: str
    category: str
    rating: float
    estimated_time: str
    cost: float
    description: str

@dataclass
class DayPlan:
    """Daily itinerary"""
    day: int
    date: str
    activities: List[Dict[str, Any]]
    estimated_cost: float
    notes: str

   

In [28]:
# Define the main State for our travel planner graph


class TravelPlannerState(TypedDict):
    """
    Complete state for the travel planner agent.
    
    This state flows through all nodes and accumulates information
    throughout the planning process.
    """
    user_request: str                    # Original user input
    travel_request: Optional[TravelRequest]  # Parsed travel request details
    flight_options: List[FlightOption]   # Available flights
    hotel_options: List[HotelOption]     # Available hotels
    weather_info: Optional[WeatherInfo]  # Weather information for the destination
    attractions: List[Attraction]        # Recommended attractions

    #### selected options and final plan ####
    selected_flight: Optional[FlightOption]  # User's chosen flight
    selected_hotel: Optional[HotelOption]    # User's chosen hotel

    ### Planning results
    itinerary: List[DayPlan]              # Final day-by-day itinerary
    total_cost: float                      # Total estimated cost of the trip
    cost_breakdown: Dict[str, float]        # Breakdown of costs (flights, hotels, activities)


    ###nprocess control
    planning_stage: str                    # Current stage of the planning process
    errors: List[str]                   # Any error messages encountered during processing
    completed_steps: List[str]              # List of completed steps in the planning process


    # Final Output
    final_output: str

    


In [29]:
# Initlize our state with default values

def create_initial_state(user_input: str) -> TravelPlannerState:
    """Create initial state from user request"""
    return TravelPlannerState(
        user_request=user_input,
        travel_request=None,
        flight_options=[],
        hotel_options=[],
        weather_info=None,
        attractions=[],
        selected_flight=None,
        selected_hotel=None,
        itinerary=[],
        total_cost=0.0,
        cost_breakdown={},
        planning_stage="initialization",
        errors=[],
        completed_steps=[],
        final_output=""
    )
    

In [19]:
from utils.utils_functions import get_llm, extract_with_regex
from langchain_core.messages import SystemMessage, HumanMessage

In [36]:
def process_user_input(state: TravelPlannerState) -> TravelPlannerState:
    """
    Parse user input and extract travel requirements using LLM or regex fallback.
    
    This node:
    1. Analyzes the user's travel request
    2. Extracts key information (dates, destination, budget, etc.)
    3. Validates the input
    4. Creates a structured TravelRequest object
    """
    print("🔍 Processing user input...")
    user_request = state['user_request']
    
    try:
        llm = get_llm()
        
        if llm:
            # Use LLM for extraction
            print("🤖 Using LLM for intelligent extraction...")
            
            system_prompt = """You are a travel planning assistant. Extract travel information from user requests.
            
            Please extract the following information and return it in JSON format:
            {
                "origin": "departure city",
                "destination": "destination city", 
                "departure_date": "YYYY-MM-DD format",
                "return_date": "YYYY-MM-DD format",
                "travelers": number_of_travelers,
                "budget": budget_amount_in_dollars,
                "preferences": ["list", "of", "preferences"],
                "trip_type": "business|leisure|adventure|family"
            }
            
            If any information is missing, make reasonable assumptions based on context.
            For dates, if not specified, assume departure is 30 days from now and return is 7 days later.
            For budget, if not specified, assume $2000 per person.
            For preferences, infer from the context (e.g., museums, local cuisine, history, nature, nightlife, shopping, architecture).
            """
            
            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=f"User request: {user_request}")
            ]
            
            response = llm.invoke(messages)
            
            # Parse JSON response
            import json
            try:
                extracted_data = json.loads(response.content)
            except json.JSONDecodeError:
                # Try to extract JSON from response if it's wrapped in markdown
                import re
                json_match = re.search(r'```json\n(.*?)\n```', response.content, re.DOTALL)
                if json_match:
                    extracted_data = json.loads(json_match.group(1))
                else:
                    raise ValueError("Could not parse LLM response as JSON")
                    
        else:
            # Fallback to regex extraction
            print("📝 Using regex-based extraction (fallback)...")
            extracted_data = extract_with_regex(user_request)
        
        # Create TravelRequest object
        travel_request = TravelRequest(
            origin=extracted_data.get('origin', 'New York'),
            destination=extracted_data.get('destination', 'Paris'),
            departure_date=extracted_data.get('departure_date', '2024-06-15'),
            return_date=extracted_data.get('return_date', '2024-06-22'),
            # travelers=extracted_data.get('travelers', 2),
            budget=float(extracted_data.get('budget', 2000.0)),
            preferences=extracted_data.get('preferences', ['museums', 'local cuisine', 'history']),
            trip_type=extracted_data.get('trip_type', 'leisure')
        )
        
        print(f"✅ Parsed request:")
        print(f"   🏙️ Origin: {travel_request.origin}")
        print(f"   🎯 Destination: {travel_request.destination}")
        print(f"   📅 Dates: {travel_request.departure_date} to {travel_request.return_date}")
        # print(f"   👥 Travelers: {travel_request.travelers}")
        print(f"   💰 Budget: ${travel_request.budget}")
        print(f"   🎭 Type: {travel_request.trip_type}")
        print(f"   🎨 Preferences: {', '.join(travel_request.preferences)}")
        
        # Create updated state
        updated_state = dict(state)
        updated_state.update({
            'travel_request': travel_request,
            'planning_stage': "flight_search",
            'completed_steps': state['completed_steps'] + ["input_processing"]
        })
        return TravelPlannerState(**updated_state)
        
    except Exception as e:
        print(f"❌ Error processing input: {e}")
        print("🔄 Falling back to sample data...")
        
        # Fallback to sample data if everything fails
        travel_request = TravelRequest(
            origin="New York",
            destination="Paris",
            departure_date="2024-06-15",
            return_date="2024-06-22",
            # travelers=2,
            budget=3000.0,
            preferences=["museums", "local cuisine", "history"],
            trip_type="leisure"
        )
        
        # Create updated state for fallback
        updated_state = dict(state)
        updated_state.update({
            'travel_request': travel_request,
            'planning_stage': "flight_search",
            'completed_steps': state['completed_steps'] + ["input_processing"],
            'errors': state['errors'] + [f"Input processing error (using fallback): {str(e)}"]
        })
        return TravelPlannerState(**updated_state)


In [37]:
print("Testing the enhanced travel request parsing with LLM and regex fallback...")

test_input = "I want to travel from London to Tokyo from June 15, 2024 to June 25, 2024. I have a budget of $4000 for 2 people. I'm interested in temples, food, and gardens."

test_state = create_initial_state(test_input)

Testing the enhanced travel request parsing with LLM and regex fallback...


In [38]:
test_state

{'user_request': "I want to travel from London to Tokyo from June 15, 2024 to June 25, 2024. I have a budget of $4000 for 2 people. I'm interested in temples, food, and gardens.",
 'travel_request': None,
 'flight_options': [],
 'hotel_options': [],
 'weather_info': None,
 'attractions': [],
 'selected_flight': None,
 'selected_hotel': None,
 'itinerary': [],
 'total_cost': 0.0,
 'cost_breakdown': {},
 'planning_stage': 'initialization',
 'errors': [],
 'completed_steps': [],
 'final_output': ''}

In [39]:
test_state = process_user_input(state=test_state)

🔍 Processing user input...
🤖 Using LLM for intelligent extraction...
✅ Parsed request:
   🏙️ Origin: London
   🎯 Destination: Tokyo
   📅 Dates: 2024-06-15 to 2024-06-25
   💰 Budget: $4000.0
   🎭 Type: leisure
   🎨 Preferences: temples, food, gardens


In [ ]:
test_input = "I want to travel from London to Tokyo from June 15, 2024 to June 25, 2024. I have a budget of $4000 for 2 people. I'm interested in temples, food, and gardens."


### How to Get and Work with TravelPlannerState

In [ ]:
print("Create Initial State:")
user_input = "I want to plan a trip to paris for 2 people with a $3000 budget"
initial_state = create_initial_state(user_input)

print()